# Low-redshift transfer-function quick start

This notebook shows how to use the complete low-redshift transfer implementation for decaying dark matter. It first constructs the heating history directly, separating prompt low-redshift heating from delayed deposition by particles injected at high redshift. It then runs ExoCLASS with Puchwein reionization, obtains the total spectral distortion, splits it into $y$ and non-$y$ pieces, and reads the exact-$y$ redshift history.

DarkAges transfer grids use $r=1+z$; ExoCLASS and the exact-$y$ history use the physical redshift $z$. The production mode is `extend`, which includes the high-high, low-low, and delayed high-to-low blocks.

## Prerequisites

The large transfer tables are distributed separately from git. Install them at the paths listed in `LOWZ_TRANSFER_TABLES.sha256`, then verify them from the repository root with:

```bash
shasum -a 256 -c LOWZ_TRANSFER_TABLES.sha256
make class
cd python && python setup.py build_ext --inplace
```

The notebook contains no machine-specific paths and can be launched from the repository root or from `notebooks/`.

In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "DarkAgesModule").is_dir() and (candidate / "source").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the ExoCLASS_perso checkout.")


ROOT = find_repository_root()
DARKAGES_BASE = ROOT / "DarkAgesModule"
os.environ["DARKAGES_BASE"] = str(DARKAGES_BASE)
sys.path.insert(0, str(DARKAGES_BASE))
sys.path.insert(0, str(ROOT / "python"))

print("Repository:", ROOT)

In [ ]:
manifest = ROOT / "LOWZ_TRANSFER_TABLES.sha256"
required_tables = []
for line in manifest.read_text().splitlines():
    if line and not line.startswith("#"):
        _, relative_path = line.split(maxsplit=1)
        required_tables.append(ROOT / relative_path)

missing = [path for path in required_tables if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing out-of-band transfer tables:\n"
        + "\n".join(str(path) for path in missing)
    )
print(f"Found all {len(required_tables)} required transfer tables.")

## 1. Inspect $f_{\rm heat}(z)$ and delayed deposition

The low-low and high-to-low bridge matrices must be convolved separately because they have different injection grids. The bridge denominator is evaluated on its own low-redshift deposition grid. `combine_lowz_heat_results` aligns and adds the two low-deposition pieces without any extra Hubble-ratio or factor-of-eight correction.

In [ ]:
import DarkAges
from DarkAges.lowz import (
    combine_lowz_heat_results,
    low_high_masks,
    validate_lowz_transfer_blocks,
)
from DarkAges.recipes import spec_elec_and_phot

# Planck 2018 background used for the draft comparison.
DarkAges.set_background(
    H0=67.66,
    Om_M=0.31104850446994281,
    Om_R=7.9111186056792255e-5,
)

HIGH_HEAT = DarkAges.transfer_functions[DarkAges.channel_dict["Heat"]]
LOW_HEAT, DELAYED_HEAT = DarkAges.get_lowz_heat_transfer_functions()
validate_lowz_transfer_blocks(LOW_HEAT, DELAYED_HEAT, HIGH_HEAT)


def injection_model(particle, mass_gev, lifetime_s, redshift):
    return spec_elec_and_phot(
        [particle],
        mass_gev,
        redshift=redshift,
        t_dec=lifetime_s,
        hist="decay",
        branchings=np.ones(1),
    )


def calculate_fheat(mass_gev, lifetime_s, particle="dirac_electron", mode="extend"):
    low_injection_mask, high_injection_mask = low_high_masks(
        LOW_HEAT.z_injected, DELAYED_HEAT.z_injected, mode
    )
    high_model = injection_model(
        particle, mass_gev, lifetime_s, HIGH_HEAT.z_injected
    )
    low_model = injection_model(
        particle, mass_gev, lifetime_s, LOW_HEAT.z_injected
    )
    deposition_model = injection_model(
        particle, mass_gev, lifetime_s, DELAYED_HEAT.z_deposited
    )

    high = high_model.calc_f(HIGH_HEAT)[-1]
    low = low_model.calc_f(
        LOW_HEAT, injection_mask=low_injection_mask
    )[-1]
    delayed = high_model.calc_f(
        DELAYED_HEAT,
        injection_mask=high_injection_mask,
        deposition_normalization=deposition_model.normalization,
    )[-1]

    low_redshift, low_total = combine_lowz_heat_results(
        LOW_HEAT, low, DELAYED_HEAT, delayed,
        low_injection_mask, high_injection_mask,
    )
    low_positions = np.searchsorted(LOW_HEAT.z_deposited, low_redshift)
    delayed_positions = np.searchsorted(
        DELAYED_HEAT.z_deposited, low_redshift
    )

    use_low, use_high = low_high_masks(
        low_redshift, HIGH_HEAT.z_deposited, mode
    )
    extended_redshift = np.concatenate(
        (low_redshift[use_low], HIGH_HEAT.z_deposited[use_high])
    )
    extended_heat = np.concatenate(
        (low_total[use_low], high[use_high])
    )
    return {
        "legacy_redshift": HIGH_HEAT.z_deposited.copy(),
        "legacy": high,
        "extended_redshift": extended_redshift,
        "extended": extended_heat,
        "low_redshift": low_redshift,
        "low_low": low[low_positions],
        "delayed": delayed[delayed_positions],
        "low_total": low_total,
    }

In [ ]:
MASS_GEV = 0.45
LIFETIME_S = 1.2e25
PARTICLE = "dirac_electron"  # change to "dirac_photon" for photons

fheat = calculate_fheat(MASS_GEV, LIFETIME_S, PARTICLE, mode="extend")


def signed_log_plot(
    axis, x, y, *, label, color, linewidth=1.8, linestyle="-", zorder=None
):
    y = np.asarray(y)
    absolute = np.abs(y)
    visible = absolute > np.nanmax(absolute) * 1.0e-10
    axis.loglog(
        x, np.where((y >= 0.0) & visible, absolute, np.nan),
        label=label, color=color, linewidth=linewidth,
        linestyle=linestyle, zorder=zorder,
    )
    axis.loglog(
        x, np.where((y < 0.0) & visible, absolute, np.nan),
        color=color, linewidth=linewidth, linestyle=":", zorder=zorder,
    )


fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
signed_log_plot(
    axes[0], fheat["extended_redshift"], fheat["extended"],
    label="complete `extend` result", color="#1f77b4", linewidth=2.2,
)
signed_log_plot(
    axes[0], fheat["legacy_redshift"], fheat["legacy"],
    label="high-$z$ table only", color="0.25",
    linestyle="--", zorder=3,
)
axes[0].set_xlabel(r"$1+z_{\rm dep}$")
axes[0].set_ylabel(r"$|f_{\rm heat}|$")
axes[0].set_title("Legacy coverage and low-$z$ extension")
axes[0].legend(frameon=False)

signed_log_plot(
    axes[1], fheat["low_redshift"], fheat["low_low"],
    label=r"low injection $\rightarrow$ low deposition", color="#2ca02c",
)
signed_log_plot(
    axes[1], fheat["low_redshift"], fheat["delayed"],
    label=r"high injection $\rightarrow$ low deposition", color="#d62728",
)
signed_log_plot(
    axes[1], fheat["low_redshift"], fheat["low_total"],
    label="sum", color="#1f77b4", linewidth=2.2,
)
axes[1].axvline(4.0, color="0.55", linestyle=":", linewidth=1.0)
axes[1].set_xlim(1.05, 5.0)
axes[1].set_xlabel(r"$1+z_{\rm dep}$")
axes[1].set_ylabel(r"$|f_{\rm heat}|$")
axes[1].set_title("Low-$z$ heating decomposition")
axes[1].legend(frameon=False, fontsize=8.5)
for axis in axes:
    axis.grid(True, which="both", linestyle=":", alpha=0.25)
fig.suptitle(
    rf"$m_\chi={MASS_GEV:g}$ GeV, $\tau={LIFETIME_S:.1e}$ s, {PARTICLE}"
)
fig.tight_layout()
plt.show()

## 2. Run ExoCLASS and split $y$ from non-$y$

For normal use, the only new CLASS input is `lowz_transfer_mode = extend`. `compute_SD_with_DarkAges = yes` adds the residual non-$y$ transfer result, while CLASS computes $y$ from the matter-temperature history generated by the heating table.

Even with `sd_only_exotic = yes`, the exact-$y$ expression sees the complete thermal history, including astrophysical reionization. Therefore an exotic $y$ signal should be formed by subtracting a matched negligible-injection baseline point by point, as below. `compute_distortions_only()` is a fast `classy` path for this exotic-only setup.

In [ ]:
if not list((ROOT / "python").glob("classy*.so")):
    raise RuntimeError(
        "Build the Python wrapper first: cd python && python setup.py build_ext --inplace"
    )

from classy import Class


def exoclass_parameters(fraction, mode="extend"):
    return {
        "output": "Sd",
        "omega_b": 0.02242,
        "omega_cdm": 0.11933,
        "H0": 67.66,
        "N_ur": 2.03351,
        "N_ncdm": 1,
        "m_ncdm": 0.06,
        "n_s": 0.9665,
        "ln10^{10}A_s": 3.047,
        "reio_parametrization": "reio_stars",
        "include_reio_stars_cooling_terms": "yes",
        "include_reio_stars_helium": "yes",
        "reio_stars_photoion_file": str(
            ROOT / "external/heating/photoion_rates_Puchwein.dat"
        ),
        "reio_stars_photoheat_file": str(
            ROOT / "external/heating/photoheat_rates_Puchwein2.dat"
        ),
        "reio_stars_helium_file": str(
            ROOT / "external/heating/xHe_DarkHistory.txt"
        ),
        "f_eff_type": "DarkAges",
        "DarkAges_mode": "built_in",
        "lowz_transfer_mode": mode,
        "DM_decay_mass": MASS_GEV,
        "DM_decay_Gamma": 1.0 / LIFETIME_S,
        "DM_decay_fraction": fraction,
        "injected_particle_spectra": PARTICLE,
        "injected_particle_branching_ratio": 1,
        "compute_SD_with_DarkAges": "yes",
        "include_DH_SMresidual_distortions": "no",
        "add_SD_to_CLASS": "yes",
        # No detector cache is needed for the sharp diagnostic basis.
        "sd_branching_approx": "sharp_sharp",
        "sd_only_exotic": "yes",
        "exact_y": "yes",
        "sd_z_min": 0.01,
        "sd_z_size": 600,
        "sd_x_min": 1.0e-2,
        "sd_x_max": 1.0e4,
        "sd_x_size": 500,
    }


def run_exoclass(fraction, mode="extend"):
    cosmology = Class()
    try:
        cosmology.set(exoclass_parameters(fraction, mode))
        cosmology.compute_distortions_only()
        return {
            "amplitudes": np.array(
                cosmology.spectral_distortion_amplitudes(), copy=True
            ),
            "spectrum": np.array(
                cosmology.spectral_distortion_output(), copy=True
            ),
            "y_history": tuple(
                np.array(values, copy=True)
                for values in cosmology.spectral_distortion_y_history()
            ),
        }
    finally:
        try:
            cosmology.struct_cleanup()
        except Exception:
            pass
        cosmology.empty()

In [ ]:
MODEL_FRACTION = 1.0
BASELINE_FRACTION = 1.0e-20

model = run_exoclass(MODEL_FRACTION)
baseline = run_exoclass(BASELINE_FRACTION)

# spectral_distortion_output columns: x, nu[GHz], total, g, y, mu, residual.
frequency_ghz = model["spectrum"][:, 1]
delta_spectrum = model["spectrum"] - baseline["spectrum"]
delta_total = delta_spectrum[:, 2]
delta_y_spectrum = delta_spectrum[:, 4]
delta_nony_spectrum = delta_total - delta_y_spectrum

model_z, model_integrand, y_branching, z_weights = model["y_history"]
base_z, base_integrand, base_branching, base_weights = baseline["y_history"]
np.testing.assert_allclose(model_z, base_z, rtol=0.0, atol=0.0)
np.testing.assert_allclose(y_branching, base_branching, rtol=0.0, atol=0.0)
np.testing.assert_allclose(z_weights, base_weights, rtol=0.0, atol=0.0)

# exact_integrand_y is 4 dy/dz. CLASS applies the y branching ratio and /4.
exotic_dy_dz = 0.25 * (model_integrand - base_integrand) * y_branching
y_from_history = float(np.dot(exotic_dy_dz, z_weights))
y_reported = float(model["amplitudes"][1] - baseline["amplitudes"][1])
np.testing.assert_allclose(y_from_history, y_reported, rtol=2.0e-10, atol=1.0e-18)


def integrate_below(redshift, integrand, z_max):
    stop = int(np.searchsorted(redshift, z_max, side="left"))
    if stop < redshift.size and redshift[stop] == z_max:
        clipped_z = redshift[: stop + 1]
        clipped_integrand = integrand[: stop + 1]
    else:
        clipped_z = np.append(redshift[:stop], z_max)
        clipped_integrand = np.append(
            integrand[:stop], np.interp(z_max, redshift, integrand)
        )
    return float(np.trapz(clipped_integrand, clipped_z))


y_below_one_plus_z_4 = integrate_below(model_z, exotic_dy_dz, z_max=3.0)
print(f"Exotic y from exact history : {y_from_history:.8e}")
print(f"Exotic y reported by CLASS : {y_reported:.8e}")
print(
    "Fraction accumulated at 1+z<4: "
    f"{abs(y_below_one_plus_z_4 / y_from_history):.3f}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
signed_log_plot(
    axes[0], frequency_ghz, delta_y_spectrum,
    label="$y$", color="#e66101", linewidth=2.0,
)
signed_log_plot(
    axes[0], frequency_ghz, delta_nony_spectrum,
    label="non-$y$", color="#5e3c99", linewidth=2.0,
)
axes[0].set_xlabel(r"$\nu$ [GHz]")
axes[0].set_ylabel(r"$|\Delta I_\nu|$ [Jy sr$^{-1}$]")
axes[0].set_title("Baseline-subtracted distortion")
axes[0].legend(frameon=False)
axes[0].text(0.02, 0.03, "dotted segments are negative", transform=axes[0].transAxes, fontsize=8)

one_plus_z = 1.0 + model_z
dy_dlog_one_plus_z = one_plus_z * exotic_dy_dz
signed_log_plot(
    axes[1], one_plus_z, dy_dlog_one_plus_z,
    label=r"$dy/d\ln(1+z)$", color="#1b9e77", linewidth=2.0,
)
axes[1].axvline(4.0, color="0.45", linestyle=":", label=r"$1+z=4$")
axes[1].set_xlim(1.0, 2.0e4)
axes[1].set_xlabel(r"$1+z$")
axes[1].set_ylabel(r"$|dy/d\ln(1+z)|$")
axes[1].set_title("Exact-$y$ redshift contribution")
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(True, which="both", linestyle=":", alpha=0.25)
fig.tight_layout()
plt.show()

## Modes and common changes

- `extend` is the recommended physical calculation. It keeps the high-redshift result in the overlap and includes delayed deposition.
- `extend-new` uses the low-redshift calculation throughout the overlap and is intended as an overlap diagnostic.
- `legacy` uses only the high-redshift tables.
- `low-only` and `low-below-four` are source-isolation diagnostics; neither includes the delayed bridge and neither should replace `extend` in production.

Change `PARTICLE` to `dirac_photon` to test photon injection. For the lowest electron benchmark, use the exact parent mass $2(m_e+5\,\mathrm{keV})=1.0319978922\,\mathrm{MeV}$; rounding it to $1.03\,\mathrm{MeV}$ moves the daughter below the transfer grid. If only $y$ is needed, set `compute_SD_with_DarkAges` to `no`; the heating history and exact-$y$ calculation are unchanged, and the run is faster.